In [ ]:
import os
import sys
import re
import nltk
import unicodedata
from nltk.corpus import stopwords
from sklearn.metrics import r2_score
import numpy as np
import matplotlib.pyplot as plt

# VARIABLES GLOBALES
tokens = []
terms = []
coll_freq = {}
max_size = 30
min_size = 2

# PROCESOS
def translate(string):
    """
    Convierte caracteres acentuados a su forma no acentuada.
    """
    return (
        unicodedata.normalize("NFD", string)
        .encode("ascii", "ignore")
        .decode("ascii")
    )

def tokenize(text):
    """
    Tokeniza el texto extrayendo solo palabras comunes (letras).
    """
    text = text.lower()
    text = translate(text)
    word_re = re.compile(r"[a-z]+(?:[-'][a-z]+)*")
    return word_re.findall(text)

def delete_stopwords(tokens_doc, stopwords_list, delete_sw):
    """
    Construye el vocabulario y las frecuencias.
    Si delete_sw=True, excluye stopwords.
    """
    global terms, coll_freq

    for token in tokens_doc:
        if len(token) >= min_size and len(token) <= max_size:
            if delete_sw == True:
                if token not in stopwords_list:
                    if token not in terms:
                        terms.append(token)
                        coll_freq[token] = 1
                    else:
                        coll_freq[token] += 1
            else:
                if token not in terms:
                    terms.append(token)
                    coll_freq[token] = 1
                else:
                    coll_freq[token] += 1

    return terms


# MAIN ------------------------------------------------------------------------------------------------
!rm -f pg2000.txt
!wget -q http://www.gutenberg.org/cache/epub/2000/pg2000.txt

file1 = open('pg2000.txt', "r", encoding="utf-8", errors="ignore")

nltk.download('stopwords')
stop_words = stopwords.words("spanish")

tokens_doc = tokenize(file1.read())
delete_sw = False
terms = delete_stopwords(tokens_doc, stop_words, delete_sw)

# Total de tokens considerados
total_tokens = sum(coll_freq.values())

# Lista de (termino, frecuencia) ordenada de mayor a menor
sorted_terms = sorted(coll_freq.items(), key=lambda x: x[1], reverse=True)

# Frecuencias reales ordenadas
list_freq = [freq for term, freq in sorted_terms]

# Ranking
y = list_freq
x = list(range(1, len(list_freq) + 1))
print("Frecuencia de ranking 1: ",y[0])

# Frecuencia teórica Zipf
y_zipf = []
y_zipf.append(y[0])
for i in range(1, len(y)):
    num = y_zipf[0] / (i + 1)
    y_zipf.append(num)

# Información general
V = len(sorted_terms)   # tamaño del vocabulario
print("Cantidad total de tokens:", total_tokens)
print("Cantidad total de términos del vocabulario:", V)

# -------------------------------------------------------------
# PARTE 1: comparación teórica vs real para 10%, 20%, 30%

percentages = [0.10, 0.20, 0.30]

print("\n--- Cobertura de tokens para 10%, 20% y 30% del vocabulario ---")

for p in percentages:
    k = int(V * p)

    # tokens cubiertos por los top-k términos
    real_tokens = sum(y[:k])
    zipf_tokens = sum(y_zipf[:k])

    real_pct_tokens = 100 * real_tokens / total_tokens
    zipf_pct_tokens = 100 * zipf_tokens / total_tokens

    print(f"\nTop {int(p*100)}% del vocabulario = {k} términos")
    print(f"  Tokens teóricos según Zipf: {zipf_tokens:.2f} ({zipf_pct_tokens:.2f}% del total)")
    print(f"  Tokens reales observados:   {real_tokens} ({real_pct_tokens:.2f}% del total)")
    print(f"  Diferencia absoluta:        {abs(real_tokens - zipf_tokens):.2f}")

# -------------------------------------------------------------
# PARTE 2: poda del vocabulario y comparación con stopwords

print("\n--- Poda del vocabulario ---")

previous_pruned_non_stopwords_set = set()

for p in percentages:
    k = int(V * p)

    # top-k términos más frecuentes que se podan
    current_pruned_terms = [term for term, freq in sorted_terms[:k]]

    # cuántos de los podados son stopwords
    current_pruned_stopwords = [term for term in current_pruned_terms if term in stop_words]
    current_pruned_non_stopwords_all = [term for term in current_pruned_terms if term not in stop_words]

    # Identifica los terminos que no son palabras vacias pertenecientes a ese rango de porcentaje.
    unique_pruned_non_stopwords = [term for term in current_pruned_non_stopwords_all if term not in previous_pruned_non_stopwords_set]

    pct_stopwords_in_pruning = 100 * len(current_pruned_stopwords) / len(current_pruned_terms)

    print(f"\nPoda del {int(p*100)}% del vocabulario ({k} términos):")
    print(f"  Stopwords dentro de la poda: {len(current_pruned_stopwords)}")
    print(f"  % de la poda que coincide con stopwords: {pct_stopwords_in_pruning:.2f}%")
    print(f"  Términos podados que NO son stopwords (total acumulado): {len(current_pruned_non_stopwords_all)}")

    # mostrar algunos ejemplos de términos podados que no son stopwords y son nuevos en este porcentaje
    print("  Algunos términos podados que no son stopwords (nuevos en este porcentaje):")
    print(" ", unique_pruned_non_stopwords[:30])

    previous_pruned_non_stopwords_set.update(current_pruned_non_stopwords_all)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Frecuencia de ranking 1:  21618
Cantidad total de tokens: 357089
Cantidad total de términos del vocabulario: 23089

--- Cobertura de tokens para 10%, 20% y 30% del vocabulario ---

Top 10% del vocabulario = 2308 términos
  Tokens teóricos según Zipf: 179895.68 (50.38% del total)
  Tokens reales observados:   305470 (85.54% del total)
  Diferencia absoluta:        125574.32

Top 20% del vocabulario = 4617 términos
  Tokens teóricos según Zipf: 194882.47 (54.58% del total)
  Tokens reales observados:   324975 (91.01% del total)
  Diferencia absoluta:        130092.53

Top 30% del vocabulario = 6926 términos
  Tokens teóricos según Zipf: 203648.60 (57.03% del total)
  Tokens reales observados:   334561 (93.69% del total)
  Diferencia absoluta:        130912.40

--- Poda del vocabulario ---

Poda del 10% del vocabulario (2308 términos):
  Stopwords dentro de la poda: 153
  % de la poda que coincide con stopwords: 6.63%
  Términos podados que NO son stopwords (total acumulado): 2155
  Algun

**¿Qué conclusión puede obtener?**

Se concluye que se cumple la ley de zipf con valores aproximados.
Ademas que los terminos con frecuencia 10000 serán 0 tanto en el valor esperado como en el real debido a que la cantidad total de terminos no es lo suficientemente grande como para tener esa cantidad de apariciones para un solo termino en el texto.